

# Trabajador del conocimiento experto

### Un asistente de respuesta a preguntas que actúa como trabajador del conocimiento experto
### Destinado a los empleados de AgroTech, una empresa de tecnología aplicada a la producción de comida

### El asistente de IA debe ser preciso y la solución debe ser de bajo coste.

Este proyecto utilizará RAG (Retrieval Augmented Generation) para garantizar que nuestro asistente de preguntas y respuestas tenga una alta precisión.

Esta primera implementación utilizará un tipo de RAG simplista, de fuerza bruta.

In [2]:
import os
import glob
from dotenv import load_dotenv
from pathlib import Path
import gradio as gr
from openai import OpenAI

In [28]:
# Setting up

load_dotenv(override=True)

MODEL = "llama3"
openai = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

### Vamos a empezar a leer todos los empleados en un diccionario y dato

In [4]:
knowledge = {}

filenames = glob.glob("../knowledge-base/empleados/*")

for filename in filenames:
    name = Path(filename).stem.split(' ')[-1]
    with open(filename, "r", encoding="utf-8") as f:
        knowledge[name.lower()] = f.read()

In [10]:
knowledge

{'castro': '# Alejandro Castro\n\n## Resumen\n\n-   **Fecha de nacimiento:** 24 de junio de 1981\n-   **Puesto:** Ingeniero Agrónomo y Director Técnico de Campo\n-   **Ubicación:** Los Llanos de Aridane, La Palma (Islas Canarias)\n-   **Salario actual:** 42.000 €\n-   **Metadatos sugeridos para ChromaDB:** `{"departamento": "tecnico_campo", "especialidad": "musaceas", "modulo_asociado": "FitoLLM"}`\n\n## Progresión de carrera en la Cooperativa\n\n-  **Marzo 2012 - Presente:** Director Técnico de Campo.\n-   Supervisa los cuadernos de campo digitales de más de 300 productores asociados de plátano.\n-   Diseña los planes de fertilización optimizada y las estrategias de control de plagas bajo normativas globales de sostenibilidad (GlobalGAP).\n-   Lidera la implantación de sensores de humedad en parcelas piloto para optimizar el consumo de agua.\n-   **Enero 2007 - Febrero 2012:** Técnico de Extensión Agraria en el Cabildo Insular.\n-   Asesoramiento directo a agricultores de medianías en

In [5]:
knowledge["lancaster"]

'# Alicia Lancaster\n## Resumen\n\n-   **Fecha de nacimiento:** 14 de marzo de 1994\n-   **Puesto:** Gestora de Cuentas B2B / Administrativa de Liquidaciones\n-   **Ubicación:** Tazacorte, La Palma (Islas Canarias)\n-   **Salario actual:** 27.000 €\n-   **Metadatos sugeridos para ChromaDB:**** `{"departamento": "administracion", "especialidad": "liquidaciones_socios", "modulo_asociado": "PrecioLLM"}`\n\n## Progresión de carrera en la Cooperativa\n\n-   **Febrero 2021 - Presente:** Gestora de Cuentas B2B.\n-   -   Encargada del cálculo y ejecución de las liquidaciones periódicas de precios a los agricultores socios en base a las categorías de fruta entregadas (Extra, Primera, Segunda).\n-   Punto de contacto principal para la resolución de dudas de los productores sobre retenciones, costes de empaquetado y anticipos de la PAC.\n-   Gestiona la facturación directa con las cadenas de distribución alimentaria (supermercados nacionales).\n\n## Historial de rendimiento anual\n\n-  **2023:** 

In [6]:
filenames = glob.glob("knowledge-base/products/*")

for filename in filenames:
    name = Path(filename).stem
    with open(filename, "r", encoding="utf-8") as f:
        knowledge[name.lower()] = f.read()

In [7]:
knowledge.keys()

dict_keys(['castro', 'herrera', 'toledo', 'lancaster', 'arcadio'])

In [19]:
SYSTEM_PREFIX = """
Representas a AgroTech, la empresa de tecnología aplicada a asistir a los agricultores.
REGLA CRÍTICA: Debes responder OBLIGATORIAMENTE en español (castellano).
Bajo ninguna circunstancia respondas en inglés.
Eres un experto en responder preguntas sobre AgroTech, sus empleados y sus productos.
Se te proporciona información adicional que podría ser relevante para la pregunta del usuario.
Da respuestas breves y precisas. Si no sabes la respuesta, dilo.

Información relevante:
"""

In [20]:
def get_relevant_context_simple(message):
    text = ''.join(ch for ch in message if ch.isalpha() or ch.isspace())
    words = text.lower().split()
    relevant_context = []
    for word in words:
        if word in knowledge:
            relevant_context.append(knowledge[word])
    return relevant_context          

## Una forma más simple

In [21]:
def get_relevant_context(message):
    text = ''.join(ch for ch in message if ch.isalpha() or ch.isspace())
    words = text.lower().split()
    return [knowledge[word] for word in words if word in knowledge]   

In [22]:
get_relevant_context("¿Quién es Alicia Lancaster?")

['# Alicia Lancaster\n## Resumen\n\n-   **Fecha de nacimiento:** 14 de marzo de 1994\n-   **Puesto:** Gestora de Cuentas B2B / Administrativa de Liquidaciones\n-   **Ubicación:** Tazacorte, La Palma (Islas Canarias)\n-   **Salario actual:** 27.000 €\n-   **Metadatos sugeridos para ChromaDB:**** `{"departamento": "administracion", "especialidad": "liquidaciones_socios", "modulo_asociado": "PrecioLLM"}`\n\n## Progresión de carrera en la Cooperativa\n\n-   **Febrero 2021 - Presente:** Gestora de Cuentas B2B.\n-   -   Encargada del cálculo y ejecución de las liquidaciones periódicas de precios a los agricultores socios en base a las categorías de fruta entregadas (Extra, Primera, Segunda).\n-   Punto de contacto principal para la resolución de dudas de los productores sobre retenciones, costes de empaquetado y anticipos de la PAC.\n-   Gestiona la facturación directa con las cadenas de distribución alimentaria (supermercados nacionales).\n\n## Historial de rendimiento anual\n\n-  **2023:**

In [23]:
get_relevant_context("Quién es Landcaster y quien es Herrera?")

['# Alejandro Herrera\n\n## Resumen\n\n-   **Fecha de nacimiento:** 11 de noviembre de 1988\n-   **Puesto:** Responsable de Control de Calidad y Jefe de Empaquetado\n-   **Ubicación:** Gáldar, Gran Canaria (Islas Canarias)\n-   **Salario actual:** 29.500 €\n-   **Metadatos sugeridos para ChromaDB:** ` {"departamento": "operaciones_almacen", "especialidad": "calibrado_dop", "modulo_asociado": "PrecioLLM"}`\n\n## Progresión de carrera en la Cooperativa\n\n-   **Septiembre 2018 - Presente:** Responsable de Control de Calidad.\n-   Supervisa las líneas de lavado, desmanado, calibrado y encajado del plátano conforme a los estándares de la Indicación Geográfica Protegida (IGP).\n-   Gestiona los turnos y la seguridad laboral de un equipo de 45 operarios de almacén durante las campañas de alta producción.\n-   Coordina la trazabilidad de la fruta desde que entra en los remolques hasta su estiba en los contenedores refrigerados.\n-   **Julio 2014 - Agosto 2018:** Inspectora de Calidad en Almac

In [24]:
def additional_context(message):
    relevant_context = get_relevant_context(message)
    if not relevant_context:
        result = "No hay ningún contexto adicional relevante para la pregunta del usuario."
    else:
        result = "El siguiente contexto adicional podría ser relevante para responder a la pregunta del usuario:\n\n"
        result += "\n\n".join(relevant_context)
    return result

In [25]:
print(additional_context("Who is Alex Lancaster?"))

El siguiente contexto adicional podría ser relevante para responder a la pregunta del usuario:

# Alicia Lancaster
## Resumen

-   **Fecha de nacimiento:** 14 de marzo de 1994
-   **Puesto:** Gestora de Cuentas B2B / Administrativa de Liquidaciones
-   **Ubicación:** Tazacorte, La Palma (Islas Canarias)
-   **Salario actual:** 27.000 €
-   **Metadatos sugeridos para ChromaDB:**** `{"departamento": "administracion", "especialidad": "liquidaciones_socios", "modulo_asociado": "PrecioLLM"}`

## Progresión de carrera en la Cooperativa

-   **Febrero 2021 - Presente:** Gestora de Cuentas B2B.
-   -   Encargada del cálculo y ejecución de las liquidaciones periódicas de precios a los agricultores socios en base a las categorías de fruta entregadas (Extra, Primera, Segunda).
-   Punto de contacto principal para la resolución de dudas de los productores sobre retenciones, costes de empaquetado y anticipos de la PAC.
-   Gestiona la facturación directa con las cadenas de distribución alimentaria 

In [26]:
def chat(message, history):
    system_message = SYSTEM_PREFIX + additional_context(message)
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

## Ahora vamos a implementarlo en Gradio utilizando la interfaz de chat -

Una forma rápida y sencilla de crear un prototipo de chat con un modelo de lenguaje grande (LLM)

In [27]:
view = gr.ChatInterface(chat, type="messages").launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
